In [1]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path

import folium
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import pandas as pd


ROOT = Path(__file__).resolve().parent
DATA_ROOT = ROOT / "Student folder" / "Student folder"
OUTPUT_ROOT = ROOT / "reports" / "eda"


@dataclass
class TripData:
    trip_id: str
    partition: str
    track: pd.DataFrame
    nodes: pd.DataFrame
    arcs: pd.DataFrame
    route: pd.DataFrame


def read_table(path: Path, names: list[str], dtypes: dict[str, str]) -> pd.DataFrame:
    return pd.read_csv(path, sep=r"\s+", names=names, dtype=dtypes, header=None)


def read_trip(trip_dir: Path, partition: str) -> TripData:
    trip_id = trip_dir.name
    return TripData(
        trip_id=trip_id,
        partition=partition,
        track=read_table(
            trip_dir / f"{trip_id}.track",
            ["longitude", "latitude", "timestamp_s"],
            {"longitude": "float64", "latitude": "float64", "timestamp_s": "float64"},
        ),
        nodes=read_table(
            trip_dir / f"{trip_id}.nodes",
            ["longitude", "latitude"],
            {"longitude": "float64", "latitude": "float64"},
        ),
        arcs=read_table(
            trip_dir / f"{trip_id}.arcs",
            ["from_node_id", "to_node_id"],
            {"from_node_id": "int64", "to_node_id": "int64"},
        ),
        route=read_table(
            trip_dir / f"{trip_id}.route",
            ["arc_id"],
            {"arc_id": "int64"},
        ),
    )


def load_partition(partition: str) -> list[TripData]:
    return [read_trip(path, partition) for path in sorted((DATA_ROOT / partition).iterdir()) if path.is_dir()]


def nonfinite_count(frame: pd.DataFrame) -> int:
    numeric = frame.select_dtypes(include="number")
    return int((~np.isfinite(numeric.to_numpy())).sum())


def connected_components(node_count: int, arcs: pd.DataFrame) -> int:
    parent = list(range(node_count))

    def find(node: int) -> int:
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    for source, target in arcs[["from_node_id", "to_node_id"]].itertuples(index=False):
        source_root = find(int(source))
        target_root = find(int(target))
        if source_root != target_root:
            parent[target_root] = source_root
    return len({find(node) for node in range(node_count)})


def collect_summaries(trips: list[TripData]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    inventory: list[dict[str, object]] = []
    quality: list[dict[str, object]] = []
    timestamps: list[dict[str, object]] = []
    networks: list[dict[str, object]] = []
    for trip in trips:
        all_frames = [trip.track, trip.nodes, trip.arcs, trip.route]
        gaps = trip.track["timestamp_s"].diff().dropna()
        invalid_coordinates = int(
            ((trip.track["longitude"].abs() > 180) | (trip.track["latitude"].abs() > 90)).sum()
        )
        invalid_arc_nodes = int(
            (
                (trip.arcs["from_node_id"] < 0)
                | (trip.arcs["to_node_id"] < 0)
                | (trip.arcs["from_node_id"] >= len(trip.nodes))
                | (trip.arcs["to_node_id"] >= len(trip.nodes))
            ).sum()
        )
        invalid_route_ids = int(((trip.route["arc_id"] < 0) | (trip.route["arc_id"] >= len(trip.arcs))).sum())
        inventory.append(
            {
                "partition": trip.partition,
                "trip_id": trip.trip_id,
                "track_rows": len(trip.track),
                "node_rows": len(trip.nodes),
                "arc_rows": len(trip.arcs),
                "route_rows": len(trip.route),
            }
        )
        quality.append(
            {
                "partition": trip.partition,
                "trip_id": trip.trip_id,
                "missing_cells": int(sum(frame.isna().sum().sum() for frame in all_frames)),
                "nonfinite_values": int(sum(nonfinite_count(frame) for frame in all_frames)),
                "duplicate_track_rows": int(trip.track.duplicated().sum()),
                "invalid_track_coordinates": invalid_coordinates,
                "invalid_arc_node_references": invalid_arc_nodes,
                "invalid_route_arc_ids": invalid_route_ids,
            }
        )
        timestamps.append(
            {
                "partition": trip.partition,
                "trip_id": trip.trip_id,
                "first_timestamp_s": trip.track["timestamp_s"].min(),
                "last_timestamp_s": trip.track["timestamp_s"].max(),
                "non_increasing_steps": int((gaps <= 0).sum()),
                "one_second_steps": int((gaps == 1).sum()),
                "gap_over_one_second": int((gaps > 1).sum()),
                "maximum_gap_s": gaps.max() if not gaps.empty else 0,
                "median_gap_s": gaps.median() if not gaps.empty else 0,
            }
        )
        degree = pd.concat([trip.arcs["from_node_id"], trip.arcs["to_node_id"]]).value_counts()
        networks.append(
            {
                "partition": trip.partition,
                "trip_id": trip.trip_id,
                "nodes": len(trip.nodes),
                "arcs": len(trip.arcs),
                "isolated_nodes": int(len(trip.nodes) - len(degree)),
                "weak_components": connected_components(len(trip.nodes), trip.arcs),
                "mean_observed_degree": degree.mean() if not degree.empty else 0,
            }
        )
    return tuple(pd.DataFrame(rows) for rows in (inventory, quality, timestamps, networks))


def arc_segments(nodes: pd.DataFrame, arcs: pd.DataFrame, limit: int | None = None) -> np.ndarray:
    selected = arcs if limit is None or len(arcs) <= limit else arcs.iloc[np.linspace(0, len(arcs) - 1, limit, dtype=int)]
    source = nodes.iloc[selected["from_node_id"].to_numpy()][["longitude", "latitude"]].to_numpy()
    target = nodes.iloc[selected["to_node_id"].to_numpy()][["longitude", "latitude"]].to_numpy()
    return np.stack([source, target], axis=1)


def make_static_plot(trip: TripData) -> None:
    fig, axis = plt.subplots(figsize=(8, 12))
    axis.add_collection(LineCollection(arc_segments(trip.nodes, trip.arcs, limit=25_000), colors="#b8bec5", linewidths=0.25, alpha=0.65))
    route_arcs = trip.arcs.iloc[trip.route["arc_id"].drop_duplicates().to_numpy()]
    axis.add_collection(LineCollection(arc_segments(trip.nodes, route_arcs), colors="#2ca02c", linewidths=1.2, alpha=0.95, label="Reference route"))
    axis.plot(trip.track["longitude"], trip.track["latitude"], color="#1f77b4", linewidth=1.0, alpha=0.8, label="Noisy GPS track")
    axis.scatter(trip.track["longitude"].iloc[0], trip.track["latitude"].iloc[0], c="#00a651", s=35, label="Start", zorder=5)
    axis.scatter(trip.track["longitude"].iloc[-1], trip.track["latitude"].iloc[-1], c="#d62728", s=35, label="End", zorder=5)
    axis.set(title=f"Trip {trip.trip_id}: GPS, Road Network and Reference Route", xlabel="Longitude", ylabel="Latitude")
    axis.legend(loc="best")
    axis.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f"trip_{trip.trip_id}_overview.png", dpi=180)
    plt.close(fig)


def make_distribution_plot(inventory: pd.DataFrame, timestamps: pd.DataFrame) -> None:
    merged = inventory.merge(timestamps[["partition", "trip_id", "maximum_gap_s"]], on=["partition", "trip_id"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for partition, group in merged.groupby("partition"):
        axes[0].hist(group["track_rows"], bins=16, alpha=0.65, label=partition)
        axes[1].hist(group["arc_rows"], bins=16, alpha=0.65, label=partition)
        axes[2].hist(group["maximum_gap_s"], bins=16, alpha=0.65, label=partition)
    axes[0].set(title="GPS observations per trip", xlabel="Rows", ylabel="Trips")
    axes[1].set(title="Road arcs per trip", xlabel="Arcs", ylabel="Trips")
    axes[2].set(title="Largest timestamp gap", xlabel="Seconds", ylabel="Trips")
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / "dataset_distributions.png", dpi=180)
    plt.close(fig)


def make_folium_map(trip: TripData) -> None:
    center = [trip.track["latitude"].mean(), trip.track["longitude"].mean()]
    route_arcs = trip.arcs.iloc[trip.route["arc_id"].drop_duplicates().to_numpy()]
    fmap = folium.Map(location=center, zoom_start=14, control_scale=True)
    network_layer = folium.FeatureGroup(name="Road network (sampled)", show=False)
    for segment in arc_segments(trip.nodes, trip.arcs, limit=8_000):
        folium.PolyLine([(segment[0][1], segment[0][0]), (segment[1][1], segment[1][0])], color="#9aa0a6", weight=1, opacity=0.45).add_to(network_layer)
    network_layer.add_to(fmap)
    route_layer = folium.FeatureGroup(name="Reference route")
    for segment in arc_segments(trip.nodes, route_arcs):
        folium.PolyLine([(segment[0][1], segment[0][0]), (segment[1][1], segment[1][0])], color="#22a447", weight=4, opacity=0.85).add_to(route_layer)
    route_layer.add_to(fmap)
    gps_layer = folium.FeatureGroup(name="Noisy GPS observations")
    for row in trip.track.iloc[::max(1, len(trip.track) // 1_000)].itertuples(index=False):
        folium.CircleMarker(location=(row.latitude, row.longitude), radius=2, color="#1976d2", fill=True, fill_opacity=0.7, popup=f"t={row.timestamp_s:g}s").add_to(gps_layer)
    gps_layer.add_to(fmap)
    folium.Marker([trip.track["latitude"].iloc[0], trip.track["longitude"].iloc[0]], tooltip="Start", icon=folium.Icon(color="green")).add_to(fmap)
    folium.Marker([trip.track["latitude"].iloc[-1], trip.track["longitude"].iloc[-1]], tooltip="End", icon=folium.Icon(color="red")).add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.save(OUTPUT_ROOT / f"trip_{trip.trip_id}_map.html")


def write_readme(inventory: pd.DataFrame, quality: pd.DataFrame, timestamps: pd.DataFrame, networks: pd.DataFrame, trip: TripData) -> None:
    partition_counts = inventory.groupby("partition")["trip_id"].count().to_dict()
    text = f"""# PathSnap EDA Results

## Dataset inventory

- Training trips: {partition_counts.get('train', 0)}
- Student-test trips: {partition_counts.get('student_test', 0)}
- Each trip contains a GPS track, road-network nodes, directed road arcs, and a reference arc route.

## Data-quality checks

- Missing cells: {int(quality['missing_cells'].sum())}
- Non-finite numeric values: {int(quality['nonfinite_values'].sum())}
- Invalid GPS coordinates: {int(quality['invalid_track_coordinates'].sum())}
- Invalid road-node references: {int(quality['invalid_arc_node_references'].sum())}
- Invalid reference-route arc IDs: {int(quality['invalid_route_arc_ids'].sum())}
- Non-increasing timestamp steps: {int(timestamps['non_increasing_steps'].sum())}
- Timestamp gaps above one second: {int(timestamps['gap_over_one_second'].sum())}

## Selected visualisation

Trip `{trip.trip_id}` from the `{trip.partition}` partition is visualised in the PNG and interactive Folium HTML map. The map uses blue for noisy GPS observations, green for the reference route, and grey for sampled road-network arcs.

## Files

- `trip_inventory.csv`: rows per raw file for every trip.
- `data_quality.csv`: missing-value, validity, and duplicate checks.
- `timestamp_summary.csv`: ordering and GPS sampling-gap checks.
- `network_summary.csv`: network size, isolated-node, and connectivity checks.
- `dataset_distributions.png`: trip-size and timing distributions.
- `trip_{trip.trip_id}_overview.png`: static geographical overview.
- `trip_{trip.trip_id}_map.html`: interactive Folium map.
"""
    (OUTPUT_ROOT / "README.md").write_text(text, encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description="Generate reproducible EDA outputs for the PathSnap dataset.")
    parser.add_argument("--trip", default="00000001", help="Training trip ID to visualise.")
    args = parser.parse_args()
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    trips = load_partition("train") + load_partition("student_test")
    inventory, quality, timestamps, networks = collect_summaries(trips)
    inventory.to_csv(OUTPUT_ROOT / "trip_inventory.csv", index=False)
    quality.to_csv(OUTPUT_ROOT / "data_quality.csv", index=False)
    timestamps.to_csv(OUTPUT_ROOT / "timestamp_summary.csv", index=False)
    networks.to_csv(OUTPUT_ROOT / "network_summary.csv", index=False)
    selected = next(trip for trip in trips if trip.partition == "train" and trip.trip_id == args.trip)
    make_distribution_plot(inventory, timestamps)
    make_static_plot(selected)
    make_folium_map(selected)
    write_readme(inventory, quality, timestamps, networks, selected)
    print(f"EDA outputs written to {OUTPUT_ROOT}")


if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined